# Model For AVG Fraud Probability in Merchants

Use Pseudo-labelling to create a new column named is_fraud, based on the average fraud probability. Since we only have 48 merchants, it will be necessary to first understand how we can label the average fraud probability and then apply it to the dataset in order to get rid of the existing NULL values for merchants.

We will use the dataset `df_transactions`, since this one has the merchants' information with the transactions.

In [2]:
import pandas as pd
import numpy as np
import sys
from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.calibration import CalibratedClassifierCV


In [3]:
sys.path.insert(0, "../scripts")
from spark_setup import get_spark
spark = get_spark()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/20 21:25:53 WARN Utils: Your hostname, CompuPau, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/20 21:25:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/pau_l/project-2/venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/20 21:25:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/20 21:25:54 WARN Utils: Service 'SparkUI' could not

In [4]:
df_transactions = spark.read.parquet("../data/curated/df_transactions")
df_transactions.show(5)

+------------+-------+--------------+------------------+--------------------+-----------+--------------+-------------------+-----+--------+------+------------------+---------------------+--------------------+--------------------+--------------------+------------+---------+----------------+------------------+--------------+------------------+-----------------------+
|merchant_abn|user_id|order_datetime|      dollar_value|            order_id|consumer_id|          name|            address|state|postcode|gender| fraud_probability|is_same_day_duplicate|       merchant_name|                tags|            category|revenue_band|take_rate|  category_group|     total_revenue|n_transactions|   avg_order_value|avg_merchant_fraud_prob|
+------------+-------+--------------+------------------+--------------------+-----------+--------------+-------------------+-----+--------+------+------------------+---------------------+--------------------+--------------------+--------------------+------------+-

In [5]:
df_transactions.count()

13614675

We have to treat create the is_fraud label. In this case, we will need to address the NULL values coming from the average fraud probability.

# Understand the Distribution and Determine a Threshold
We identify a threshold using the distribution of the merchants and pick a point where we can separate cases with or without fraud, while still having a reasonable number of merchants in each group. At this point, we leave Null values untouched.

In [6]:
# take all variables related to the merchants and see them individually
merchant_level = df_transactions.select(
    "merchant_abn", "category", "take_rate",
    "avg_order_value", "n_transactions", "total_revenue", "avg_merchant_fraud_prob"
).dropDuplicates(["merchant_abn"])

n_total = merchant_level.count()
n_null = merchant_level.filter(F.col("avg_merchant_fraud_prob").isNull()).count()
n_known = n_total - n_null

print(f"Merchants total: {n_total}")
print(f"With avg_merchant_fraud_prob: {n_known} ({n_known/n_total:.1%})")
print(f"NULL: {n_null} ({n_null/n_total:.1%})")

pdf_merchant = merchant_level.toPandas()

/home/pau_l/project-2/venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Merchants total: 4026
With avg_merchant_fraud_prob: 48 (1.2%)
NULL: 3978 (98.8%)


As was seen during the preprocessing, there are only 48 merchants with a recorded fraud probability.

In [7]:
# labeled stands for the merchants with fraud probability
labeled = pdf_merchant[pdf_merchant["avg_merchant_fraud_prob"].notna()].copy()

# find high fraud probabilities by sorting and finding the gaps between the distribution
pdf_sorted = labeled.sort_values("avg_merchant_fraud_prob").reset_index(drop=True)
pdf_sorted["gap_to_next"] = pdf_sorted["avg_merchant_fraud_prob"].diff()

top_gaps = pdf_sorted.nlargest(6, "gap_to_next")
for idx in top_gaps.index:
    val_after = pdf_sorted.loc[idx, "avg_merchant_fraud_prob"]
    val_before = pdf_sorted.loc[idx - 1, "avg_merchant_fraud_prob"]
    gap = pdf_sorted.loc[idx, "gap_to_next"]
    n_low, n_high = idx, len(pdf_sorted) - idx
    print(f"{val_before:6.2f} -> {val_after:6.2f}  (gap={gap:5.2f})   groups: {n_low} vs {n_high}")

 18.21 ->  28.50  (gap=10.29)   groups: 1 vs 47
 80.80 ->  89.80  (gap= 9.00)   groups: 46 vs 2
 53.29 ->  61.92  (gap= 8.64)   groups: 31 vs 17
 72.73 ->  80.80  (gap= 8.07)   groups: 45 vs 3
 44.01 ->  48.67  (gap= 4.66)   groups: 29 vs 19
 48.67 ->  53.29  (gap= 4.62)   groups: 30 vs 18


The point where we can separate the fraud probability from low to high, while being able to capture a reasonable amount of merchants, is between 53.29 and 61.92, since the groups are split into 31 merchants with no fraud and 17 with fraud.

In [8]:
# threshold between the two largest gaps already established
threshold_merchant = (53.286933 + 61.923809) / 2 

# we keep NULL values as NUll and apply the threshold
pdf_merchant["is_fraud"] = np.where(
    pdf_merchant["avg_merchant_fraud_prob"].isna(),
    np.nan,
    (pdf_merchant["avg_merchant_fraud_prob"] > threshold_merchant).astype(float)
)

print(pdf_merchant["is_fraud"].value_counts())
print(f"% fraud: {pdf_merchant['is_fraud'].mean():.1%}")

is_fraud
0.0    31
1.0    17
Name: count, dtype: int64
% fraud: 35.4%


# Train/test split

In [9]:
# separate the ones that have label and the ones that don't have
seed_set = pdf_merchant[pdf_merchant["is_fraud"].notna()].copy()
unlabeled = pdf_merchant[pdf_merchant["is_fraud"].isna()].copy() # we're going to try to label this values.

print(f"Seed set (with label): {len(seed_set)}")
print(f"Unlabeled (predict): {len(unlabeled)}")

# use a stratified split with only the labeled merchants
train, test = train_test_split(
    seed_set, test_size=0.3, random_state=42, stratify=seed_set["is_fraud"]
)

print(f"Train set: {len(train)} samples, {train['is_fraud'].mean():.1%} fraud")
print(f"Test set: {len(test)} samples, {test['is_fraud'].mean():.1%} fraud")

Seed set (with label): 48
Unlabeled (predict): 3978
Train set: 33 samples, 36.4% fraud
Test set: 15 samples, 33.3% fraud


The train and the test set have 33 samples that are fraud, and the test set has 15.

# Feature Selection
With 33 rows to train the dataset, we will use Logistics Regression instead of Random Forest, since this model could overfit easily.


**Feature Selection**
`total_revenue` and `n_transactions` are log-transformed, so that the scale doesn't dominate the other features.

We will exclude `category` since there are some categories with very few merchants, making it more difficult to generalise per category.

In [10]:
# apply log scale to total_revenue and n_transaction
pdf_merchant["log_total_revenue"] = np.log1p(pdf_merchant["total_revenue"])
pdf_merchant["log_n_transactions"] = np.log1p(pdf_merchant["n_transactions"])

# re-slice seed_set / unlabeled now that the log columns exist
seed_set = pdf_merchant[pdf_merchant["is_fraud"].notna()].copy()
unlabeled = pdf_merchant[pdf_merchant["is_fraud"].isna()].copy()

num_cols = ["take_rate", "avg_order_value", "log_total_revenue", "log_n_transactions"]

preprocess = ColumnTransformer([("num", StandardScaler(), num_cols)])

# pipline for model
model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000))
])

# Validating Model
We repeat the split 30 times with different seeds to see if the prediction AUC holds.

In [ ]:
n_repeats = 30
baseline_aucs = []

# fit the model
for seed in range(n_repeats):
    tr, te = train_test_split(seed_set, test_size=0.3, random_state=seed, stratify=seed_set["is_fraud"])
    m = clone(model)
    m.fit(tr[num_cols], tr["is_fraud"])
    auc = roc_auc_score(te["is_fraud"], m.predict_proba(te[num_cols])[:, 1])
    baseline_aucs.append(auc)

baseline_aucs = np.array(baseline_aucs)
print(f"Baseline AUC (30 splits): {baseline_aucs.mean():.3f} +/- {baseline_aucs.std():.3f}")


Baseline AUC (30 splits): 0.941 +/- 0.060


On average, the AUC is 0.941, however, this high result needs to be addressed since it could be misleading.

# Pseudo-labelling
We first train the model, and we fit it to the unlabeled set. After this, we only take the values where the model is most confident, and we retrain the model, but now using the predictions made from the unlabeled set. We then record the results to show that performing pseudo-labelling iteratively can actually improve the performance of the model.

In [12]:
max_add_per_round = 10

results_before, results_after = [], []

for seed in range(n_repeats):
    # train the model with the original train set
    tr, te = train_test_split(seed_set, test_size=0.3, random_state=seed, stratify=seed_set["is_fraud"])

    m0 = clone(model)
    # fit the test set
    m0.fit(tr[num_cols], tr["is_fraud"])

    # calculate the threshold based on the distribution of results
    pool_probs = m0.predict_proba(unlabeled[num_cols])[:, 1]
    low_threshold = np.percentile(pool_probs, 10)   # p10
    high_threshold = np.percentile(pool_probs, 90)  # p90

    # find the ROC-AUC score for the test set
    auc_before = roc_auc_score(te["is_fraud"], m0.predict_proba(te[num_cols])[:, 1])

    # take the unlabeled set and predict the values using the model trained with the tain set
    pool = unlabeled.copy()
    pool["pred_proba"] = m0.predict_proba(pool[num_cols])[:, 1]

    # take the values that are most confident to be fraud and to not be fraud.
    conf_fraud = pool[pool["pred_proba"] >= high_threshold].nlargest(max_add_per_round, "pred_proba").assign(is_fraud=1.0)
    conf_clean = pool[pool["pred_proba"] <= low_threshold].nsmallest(max_add_per_round, "pred_proba").assign(is_fraud=0.0)
    new_labels = pd.concat([conf_fraud, conf_clean]).drop(columns=["pred_proba"])

    # retrain the model but with the pseudo-labels 
    tr_aug = pd.concat([tr, new_labels], axis=0)
    m1 = clone(model)
    m1.fit(tr_aug[num_cols], tr_aug["is_fraud"])
    # find ROC-AUC for the new model
    auc_after = roc_auc_score(te["is_fraud"], m1.predict_proba(te[num_cols])[:, 1])  # same untouched test fold

    results_before.append(auc_before)
    results_after.append(auc_after)

results_before = np.array(results_before)
results_after = np.array(results_after)

print(f"Before pseudo-labels: {results_before.mean():.3f} +/- {results_before.std():.3f}")
print(f"After  pseudo-labels: {results_after.mean():.3f} +/- {results_after.std():.3f}")
print(f"Splits where it improved: {(results_after > results_before).mean():.1%}")

diff = results_after - results_before


print(f"Mean difference: {diff.mean():.4f}")
print(f"Std of difference: {diff.std():.4f}")
print(f"Splits worsened:  {(diff < 0).mean():.1%}")
print(f"Splits unchanged: {(diff == 0).mean():.1%}")

Before pseudo-labels: 0.941 +/- 0.060
After  pseudo-labels: 0.943 +/- 0.059
Splits where it improved: 13.3%
Mean difference: 0.0013
Std of difference: 0.0088
Splits worsened:  6.7%
Splits unchanged: 80.0%


In this case, we can observe that using the pseudo-labels for the training set doesn't improve the results compared to training without them, suggesting that regardless of the training set we have, the results will continue to be the same. This could be due to the effect that each feature has on the model.

In [13]:
X_num = seed_set[num_cols].dropna()
vif_df = pd.DataFrame({
    "feature": num_cols,
    "VIF": [variance_inflation_factor(X_num.values, i) for i in range(len(num_cols))]
})
print(vif_df)

              feature       VIF
0           take_rate  1.082217
1     avg_order_value  3.402466
2   log_total_revenue  2.575093
3  log_n_transactions  5.386031


In [ ]:
# check correlation of features
corr_check = seed_set[["avg_merchant_fraud_prob"] + num_cols].corr()
print(corr_check["avg_merchant_fraud_prob"].sort_values())


log_total_revenue         -0.635518
log_n_transactions        -0.392851
take_rate                 -0.164761
avg_order_value           -0.158892
avg_merchant_fraud_prob    1.000000
Name: avg_merchant_fraud_prob, dtype: float64


Total revenue has a negative correlation of 0.6 with average merchant fraud probability. Thus, merchants with higher revenue tend to have a lower merchant fraud probability.

In [15]:
# exploring category to determine if it should have been placed as a feature
print(seed_set["category"].value_counts())

category
antique shops - sales, repairs, and restoration services                                 10
jewelry, watch, clock, and silverware shops                                               7
furniture, home furnishings and equipment shops, and manufacturers, except appliances     4
art dealers and galleries                                                                 2
artist supply and craft shops                                                             2
health and beauty spas                                                                    2
computers, computer peripheral equipment, and software                                    2
shoe shops                                                                                2
florists supplies, nursery stock, and flowers                                             2
tent and awning shops                                                                     2
telecom                                                                

Having categories with only one or two is insufficient for the model to be able to generalise this category. Thus, this feature should not be considered.

We can observe that, in general, the model learns mostly from the features of total revenue and the number of transactions, while the others don't have an effect on determining whether something is fraud or not. Moreover, since it was seen that doing the pseudo-labelling recursively can lead to worse results, we will only fit the model once and see the results.

In [16]:
model.fit(seed_set[num_cols], seed_set["is_fraud"])  # fit once on the FULL labeled set — no self-training

base_rate = seed_set["is_fraud"].mean()
# predict the model for merchants with no label
raw_pool_mean = model.predict_proba(unlabeled[num_cols])[:, 1].mean()
print(f"Real fraud rate in seed set: {base_rate:.1%}")
print(f"Raw (uncalibrated) mean prediction on pool: {raw_pool_mean:.1%}")


Real fraud rate in seed set: 35.4%
Raw (uncalibrated) mean prediction on pool: 74.7%


In [17]:
calibrated_model = CalibratedClassifierCV(model, method="sigmoid", cv=5)
calibrated_model.fit(seed_set[num_cols], seed_set["is_fraud"])

unlabeled["pred_proba"] = calibrated_model.predict_proba(unlabeled[num_cols])[:, 1]
print(f"Calibrated mean prediction on pool: {unlabeled['pred_proba'].mean():.1%}")

Calibrated mean prediction on pool: 67.6%


In [18]:
compare = pd.DataFrame({
    "labeled_mean": seed_set[num_cols].mean(),
    "labeled_median": seed_set[num_cols].median(),
    "population_mean": pdf_merchant[num_cols].mean(),
    "population_median": pdf_merchant[num_cols].median(),
})
print(compare)

print()
for col in num_cols:
    pct = (pdf_merchant[col] < seed_set[col].median()).mean()
    print(f"{col}: the labeled median sits at the {pct:.0%} percentile of the full population")


                    labeled_mean  labeled_median  population_mean  \
take_rate               4.173958        4.400000         4.397576   
avg_order_value      6541.825045      734.726740      1159.595793   
log_total_revenue      13.980801       14.124302        11.908246   
log_n_transactions      6.853283        6.307037         6.062596   

                    population_median  
take_rate                    4.500000  
avg_order_value            317.329914  
log_total_revenue           11.859341  
log_n_transactions           6.037871  

take_rate: the labeled median sits at the 48% percentile of the full population
avg_order_value: the labeled median sits at the 73% percentile of the full population
log_total_revenue: the labeled median sits at the 93% percentile of the full population
log_n_transactions: the labeled median sits at the 56% percentile of the full population


Once we calibrate the model, we will determine if an unlabeled row is fraud or not by only considering the results where the model is most confident. 

In [19]:
p10 = seed_set[num_cols].quantile(0.10) # 10% lowest values
p90 = seed_set[num_cols].quantile(0.90) # 10% highest values

# take the columns were the model is more certain of the results
well_supported = pd.Series(True, index=unlabeled.index)
for col in num_cols:
    well_supported &= (unlabeled[col] >= p10[col])

print(f"Merchants in a well-supported range: {well_supported.sum()} / {len(unlabeled)}")

unlabeled_final = unlabeled[well_supported].copy()


Merchants in a well-supported range: 1856 / 3978


In [20]:
# join the whole dataset with the labels that the model was more certain to be correct.
pdf_merchant = pdf_merchant.merge(
    unlabeled_final[["merchant_abn", "pred_proba"]], on="merchant_abn", how="left"
)

pdf_merchant["merchant_fraud_score_final"] = np.where(
    pdf_merchant["avg_merchant_fraud_prob"].notna(),
    pdf_merchant["avg_merchant_fraud_prob"] / 100,   # normalise when the merchant already had a fraud probability
    pdf_merchant["pred_proba"]    # use the prediction of the model            
)


n_final_known = pdf_merchant["merchant_fraud_score_final"].notna().sum()
print(f"Merchants with fraud score (real + pseudo-label): {n_final_known} / {len(pdf_merchant)}")
print(f"NULL: {pdf_merchant['merchant_fraud_score_final'].isna().sum()}")

pdf_merchant["merchant_fraud_score_final"].describe()

Merchants with fraud score (real + pseudo-label): 1904 / 4026
NULL: 2122


count    1904.000000
mean        0.568832
std         0.178305
min         0.100760
25%         0.424141
50%         0.576702
75%         0.719318
max         0.910961
Name: merchant_fraud_score_final, dtype: float64

In [ ]:
threshold_merchant_normalized = threshold_merchant / 100   

# convert the score in the is_fraud label
# use the threshold to assign the binary values
# keep the Nan were the model wasn't able to determine the results
pdf_merchant["is_fraud_merchant"] = np.where(
    pdf_merchant["merchant_fraud_score_final"].isna(),
    np.nan,
    (pdf_merchant["merchant_fraud_score_final"] > threshold_merchant_normalized).astype(float)
)

print(pdf_merchant["is_fraud_merchant"].value_counts(dropna=False))

is_fraud_merchant
NaN    2122
1.0     955
0.0     949
Name: count, dtype: int64


Although the model wasn't able to categorise all the records, it was still possible to find more results for the merchants. It is important to consider that the model's performance was affected by the limited number of merchants with fraud and also by the features, since only two of them were able to determine the behaviour of the is_fraud column.

In [22]:
merchant_fraud_output = pdf_merchant[[
    "merchant_abn", "merchant_fraud_score_final", "is_fraud_merchant"
]].copy()

merchant_fraud_output.to_parquet("../data/curated/merchant_fraud_labels.parquet", index=False)
print(f"Stored: {len(merchant_fraud_output)} merchants")

Stored: 4026 merchants
